In [ ]:
%livy.pyspark

from pyspark.sql.functions import col
from pyspark.sql.functions import when, lower
from pyspark.sql.types import *
import json



def cast_df_to_normalized_schema(df):
    normalized_fields = []

    for f in df.schema.fields:
        normalized_fields.append(
            col(f.name).cast(normalize_type(f.dataType)).alias(f.name)
        )

    return df.select(normalized_fields)


def spark_type_to_hive(dt):
    """
    Convert Spark DataType -> Hive DDL type
    Support nested: array / struct / map
    """

    # ===== PRIMITIVE =====
    if isinstance(dt, StringType):
        return "STRING"

    if isinstance(dt, BooleanType):
        return "BOOLEAN"

    if isinstance(dt, (ByteType, ShortType, IntegerType, LongType)):
        return "BIGINT"

    if isinstance(dt, (FloatType, DoubleType, DecimalType)):
        return "DOUBLE"

    if isinstance(dt, TimestampType):
        return "TIMESTAMP"

    if isinstance(dt, DateType):
        return "DATE"

    # ===== ARRAY =====
    if isinstance(dt, ArrayType):
        return "ARRAY<{}>".format(spark_type_to_hive(dt.elementType))

    # ===== MAP =====
    if isinstance(dt, MapType):
        v1 = spark_type_to_hive(dt.keyType)
        v2 = spark_type_to_hive(dt.valueType)
        return "MAP<{},{}>".format(v1, v2)

    # ===== STRUCT =====
    if isinstance(dt, StructType):
        fields_ddl = []
        for f in dt.fields:
            field_type = spark_type_to_hive(f.dataType)
            fields_ddl.append("{}:{}".format(f.name, field_type))
        return "STRUCT<{}>".format(",".join(fields_ddl))

    # ===== FALLBACK =====
    return "STRING"


def spark_schema_to_hive_columns(schema):
    cols = []
    for f in schema.fields:
        hive_type = spark_type_to_hive(f.dataType)
        cols.append("`{}` {}".format(f.name, hive_type))
    return ",\n".join(cols)


def normalize_type(dt):
    """
    Chuẩn hoá Spark DataType, support:
    - primitive
    - array
    - struct
    """

    # ===== PRIMITIVE =====
    if isinstance(dt, StringType):
        return StringType()

    if isinstance(dt, BooleanType):
        return BooleanType()

    if isinstance(dt, (ByteType, ShortType, IntegerType, LongType)):
        return LongType()

    if isinstance(dt, (FloatType, DoubleType, DecimalType)):
        return DoubleType()

    if isinstance(dt, TimestampType):
        return TimestampType()

    if isinstance(dt, DateType):
        return DateType()

    # ===== ARRAY =====
    if isinstance(dt, ArrayType):
        return ArrayType(normalize_type(dt.elementType), containsNull=dt.containsNull)

    # ===== STRUCT =====
    if isinstance(dt, StructType):
        new_fields = []
        for f in dt.fields:
            new_fields.append(
                StructField(f.name, normalize_type(f.dataType), f.nullable)
            )
        return StructType(new_fields)

    # ===== MAP (ít gặp nhưng vẫn xử) =====
    if isinstance(dt, MapType):
        return MapType(
            normalize_type(dt.keyType),
            normalize_type(dt.valueType),
            dt.valueContainsNull,
        )

    # ===== FALLBACK =====
    return StringType()


def spark_type_to_json(dt):
    # ===== PRIMITIVE =====
    if isinstance(dt, StringType):
        return {"type": "string"}

    if isinstance(dt, BooleanType):
        return {"type": "boolean"}

    if isinstance(dt, (ByteType, ShortType, IntegerType, LongType)):
        return {"type": "long"}

    if isinstance(dt, (FloatType, DoubleType, DecimalType)):
        return {"type": "double"}

    if isinstance(dt, TimestampType):
        return {"type": "timestamp"}

    if isinstance(dt, DateType):
        return {"type": "date"}

    # ===== ARRAY =====
    if isinstance(dt, ArrayType):
        return {"type": "array", "elementType": spark_type_to_json(dt.elementType)}

    # ===== MAP =====
    if isinstance(dt, MapType):
        return {
            "type": "map",
            "keyType": spark_type_to_json(dt.keyType),
            "valueType": spark_type_to_json(dt.valueType),
        }

    # ===== STRUCT =====
    if isinstance(dt, StructType):
        return {
            "type": "struct",
            "fields": [
                {
                    "name": f.name,
                    "nullable": f.nullable,
                    "dataType": spark_type_to_json(f.dataType),
                }
                for f in dt.fields
            ],
        }

    # ===== FALLBACK =====
    return {"type": "string"}


def spark_schema_to_json(schema):
    return {
        "type": "struct",
        "fields": [
            {
                "name": f.name,
                "nullable": f.nullable,
                "dataType": spark_type_to_json(f.dataType),
            }
            for f in schema.fields
        ],
    }


def json_type_to_pyarrow(jt):
    import pyarrow as pa
    t = jt["type"]

    # ===== PRIMITIVE =====
    if t == "string":
        return pa.string()

    if t == "boolean":
        return pa.bool_()

    if t == "long":
        return pa.int64()

    if t == "double":
        return pa.float64()

    if t == "timestamp":
        return pa.timestamp("ms")

    if t == "date":
        return pa.date32()

    # ===== ARRAY =====
    if t == "array":
        return pa.list_(json_type_to_pyarrow(jt["elementType"]))

    # ===== MAP =====
    if t == "map":
        return pa.map_(
            json_type_to_pyarrow(jt["keyType"]), json_type_to_pyarrow(jt["valueType"])
        )

    # ===== STRUCT =====
    if t == "struct":
        return pa.struct(
            [
                pa.field(
                    f["name"],
                    json_type_to_pyarrow(f["dataType"]),
                    f.get("nullable", True),
                )
                for f in jt["fields"]
            ]
        )

    # ===== FALLBACK =====
    return pa.string()


def load_pyarrow_schema_from_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        schema_json = json.load(f)

    assert schema_json["type"] == "struct", "Root schema must be struct"

    fields = []
    for f in schema_json["fields"]:
        fields.append(
            pa.field(
                f["name"], json_type_to_pyarrow(f["dataType"]), f.get("nullable", True)
            )
        )

    return pa.schema(fields)

In [ ]:
%livy.pyspark
from pyspark.sql.functions import lit

def align_columns(df, all_columns):
    """
    Thêm các cột còn thiếu vào df, gán giá trị null
    """
    for c in all_columns:
        if c not in df.columns:
            df = df.withColumn(c, lit(None))
    # Sắp xếp lại cột cho đồng nhất
    df = df.select(*all_columns)
    return df

In [ ]:
%livy.pyspark
input_path = "/opt/datasets/crawlers/vcs/freshdesk/data/fact_cso_tickets/*"
files = list_parquet_files_hdfs("/opt/datasets/crawlers/vcs/freshdesk/data/fact_cso_tickets/")
print("Found {} Parquet files on HDFS".format(len(files)))

cols_to_cast=["cf_number_of_due_date_changes"]
dfs = []
for f in files:
    df = spark.read.parquet(f)
    for c in cols_to_cast:
        if c in df.columns:
            df = df.withColumn(c, col(c).cast("string"))
    dfs.append(df)


In [ ]:
%livy.pyspark
import os
hdfs_output = "/opt/datasets/crawlers/vcs/freshdesk/data_jsonl/fact_cso_tickets_jsonl"
parquet_files = list_parquet_files_hdfs("/opt/datasets/crawlers/vcs/freshdesk/data/fact_cso_tickets/")

print("Found {} Parquet files on HDFS".format(len(files)))

# Đọc từng file và convert sang JSONL
for f in parquet_files:
    df = spark.read.parquet(f)
    # Convert mỗi file thành 1 JSONL file trên HDFS
    file_name = os.path.basename(f).replace(".parquet", ".jsonl")
    output_path = os.path.join(hdfs_output, file_name)
    
    # ghi theo JSON Lines (one record per line)
    df.coalesce(1).write.mode("overwrite").option("compression", "none").json(output_path)
    
    print("Converted {} → {}".format(f,output_path ))

In [ ]:
%livy.pyspark
df_json = spark.read.json("/opt/datasets/crawlers/vcs/freshdesk/data_jsonl/fact_cso_tickets_jsonl/*/*")

In [ ]:
%livy.pyspark
df_json.repartition(1).write.mode("overwrite").option("compression", "none").json("/opt/datasets/crawlers/vcs/freshdesk/data_jsonl/fact_cso_tickets_jsonl_single")

In [ ]:
%livy.pyspark
schema = df_json.schema

normalized_cols = []

for field in schema:
    target_type = normalize_type(field.dataType)
    normalized_cols.append(col(field.name).cast(target_type).alias(field.name))

columns_ddl = []
for field in df_json.schema:
    hive_type = spark_type_to_hive(field.dataType)
    columns_ddl.append("`{}` {}".format(field.name,hive_type ))

ddl = ",\n".join(columns_ddl)
table_name = "ods.fact_cso_tickets"
location = "output_path"

print(
    """
DROP TABLE IF EXISTS {}
""".format(table_name)
)

print(
    """
CREATE TABLE {} (
{}
)
USING PARQUET
LOCATION '{}'
""".format(table_name, ddl, location)
)



In [ ]:
%livy.pyspark
schema_json = spark_schema_to_json(df_json.schema)

schema_path = "/opt/datasets/crawlers/vcs/freshdesk/data_jsonl/fact_cso_tickets_schema3.json"

spark.sparkContext.parallelize(
    [json.dumps(schema_json, ensure_ascii=False, indent=2)]
).coalesce(1).saveAsTextFile(schema_path)


In [ ]:
%livy.pyspark
df_json.coalesce(5).write \
    .mode("overwrite") \
    .parquet("/opt/datasets/crawlers/vcs/freshdesk/data_parquet/fact_cso_tickets_parquet")